In [1]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt


def coords_to_state(row, col, width=10):
    # Formula: Multiply row by width, then add the column
    return row * width + col


def state_to_coords(state, width=10):
    # Formula: Use floor division for row, modulo for column
    row = state // width
    col = state % width
    return row, col


# Quick sanity check of the flattening helpers
print(coords_to_state(1, 2))   # Expected output: 12
print(state_to_coords(12))     # Expected output: (1, 2)


class BeginnerGridWorld:
    def __init__(self):
        self.height = 10
        self.width = 10
        self.num_states = 100
        self.num_actions = 4  # 0: Up, 1: Down, 2: Left, 3: Right
        self.goal = 99  # Bottom-right
        self.trap = 55  # Center tile
        self.P = np.zeros((100, 4, 100))
        self.R = np.zeros((100, 4))
        self.build_world()

    def build_world(self):
        for s in range(self.num_states):
            if s == self.goal or s == self.trap:
                # Terminal states: absorbing loop
                for a in range(self.num_actions):
                    self.P[s, a, s] = 1.0
                continue

            r, c = s // 10, s % 10
            # Map out coordinates if we went Up, Down, Left, or Right
            neighbors = {
                0: (max(r - 1, 0), c),        # Up (prevent leaving top wall)
                1: (min(r + 1, 9), c),        # Down (prevent leaving bottom wall)
                2: (r, max(c - 1, 0)),        # Left (prevent leaving left wall)
                3: (r, min(c + 1, 9))         # Right (prevent leaving right wall)
            }

            for a in range(self.num_actions):
                # Target states for each action
                intended_state = neighbors[a][0] * 10 + neighbors[a][1]

                # Assign 85% probability to intended move
                self.P[s, a, intended_state] += 0.85

                # Assign 5% probability to each slip direction
                for slip_action in range(self.num_actions):
                    if slip_action != a:
                        slip_state = neighbors[slip_action][0] * 10 + neighbors[slip_action][1]
                        self.P[s, a, slip_state] += 0.05

                # Define Reward: If a transition lands in the Goal, +10. Trap, -10. Else, -0.1 step cost.
                for s_next in range(self.num_states):
                    prob = self.P[s, a, s_next]
                    if prob > 0:
                        if s_next == self.goal:
                            self.R[s, a] += prob * 10.0
                        elif s_next == self.trap:
                            self.R[s, a] += prob * -10.0
                        else:
                            self.R[s, a] += prob * -0.1

    def solve_value_iteration(self, gamma=0.99, theta=1e-8, max_iters=1000):
        """
        Run the iterative loop to find the optimal values and directions.
        Returns V, policy, number of iterations taken, and the final delta.
        """
        V = np.zeros(self.num_states)
        iters_taken = max_iters
        final_delta = None
        for i in range(max_iters):
            # Compute Q-values for all states/actions in one shot:
            # Q[s, a] = R[s, a] + gamma * sum_s' P[s, a, s'] * V[s']
            Q = self.R + gamma * np.tensordot(self.P, V, axes=(2, 0))

            # Best achievable value at each state = best action's Q-value
            V_new = np.max(Q, axis=1)

            # Check if values stopped changing significantly
            delta = np.max(np.abs(V_new - V))
            V = V_new
            final_delta = delta
            if delta < theta:
                iters_taken = i + 1
                break

        # Compute the final optimal actions (the index of the maximum Q-value)
        final_Q = self.R + gamma * np.tensordot(self.P, V, axes=(2, 0))
        optimal_policy = np.argmax(np.round(final_Q, 9), axis=1)
        return V, optimal_policy, iters_taken, final_delta

    def solve_policy_iteration(self, gamma=0.99, max_iters=100):
        """
        Run Policy Iteration using SciPy exact linear system solver.
        Returns V, policy, and the number of steps taken to converge.
        """
        # Start with a policy of pointing UP (0) everywhere
        policy = np.zeros(self.num_states, dtype=int)
        V = np.zeros(self.num_states)
        state_indices = np.arange(self.num_states)
        steps_taken = max_iters

        for i in range(max_iters):
            # 1. Exact Policy Evaluation: Solve (I - gamma * P_pi) * V = R_pi
            # Slicing tip: select transition probabilities for actions chosen by our policy
            P_pi = self.P[state_indices, policy, :]
            R_pi = self.R[state_indices, policy]

            # Construct matrix A = I - gamma * P_pi
            A = np.eye(self.num_states) - gamma * P_pi

            # Solve the exact linear system using scipy.linalg.solve
            V = scipy.linalg.solve(A, R_pi)

            # 2. Policy Improvement: Pick the best actions using our new exact values
            Q = self.R + gamma * np.tensordot(self.P, V, axes=(2, 0))
            new_policy = np.argmax(np.round(Q, 9), axis=1)

            # Check if policy map has stabilized (stopped changing)
            if np.array_equal(new_policy, policy):
                steps_taken = i + 1
                break
            policy = new_policy

        return V, policy, steps_taken


ARROW = {0: '\u2191', 1: '\u2193', 2: '\u2190', 3: '\u2192'}  # Up, Down, Left, Right


def draw_value_and_policy(ax, fig, V, policy, goal, trap, title):
    grid_V = V.reshape(10, 10)
    im = ax.imshow(grid_V, cmap='viridis')
    fig.colorbar(im, ax=ax, label='State Value', fraction=0.046, pad=0.04)

    for s in range(100):
        r, c = s // 10, s % 10
        if s == goal:
            ax.text(c, r, 'G', ha='center', va='center', color='white',
                     fontsize=10, fontweight='bold')
        elif s == trap:
            ax.text(c, r, 'T', ha='center', va='center', color='red',
                     fontsize=10, fontweight='bold')
        else:
            ax.text(c, r, ARROW[policy[s]], ha='center', va='center',
                     color='white', fontsize=9)

    ax.set_title(title)
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))


if __name__ == "__main__":
    world = BeginnerGridWorld()

    gammas = [0.9, 0.95, 0.99, 0.999]
    results = []

    for gamma in gammas:
        print(f"--- Running Dynamic Programming for Gamma = {gamma} ---")

        V_vi, policy_vi, vi_iters, vi_delta = world.solve_value_iteration(gamma=gamma)
        print(f"Value Iteration converged in {vi_iters} iterations. Final delta: {vi_delta:.2e}")

        V_pi, policy_pi, pi_steps = world.solve_policy_iteration(gamma=gamma)
        print(f"Policy Iteration converged in {pi_steps} steps (Policy is stable).")

        discrepancy = np.max(np.abs(V_vi - V_pi))
        print(f"Max Value discrepancy between VI and PI: {discrepancy:.2e}")
        print()

        results.append((gamma, V_vi, policy_vi))

    # Single combined figure: one value/policy map per gamma
    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    for ax, (gamma, V_vi, policy_vi) in zip(axes.flat, results):
        draw_value_and_policy(ax, fig, V_vi, policy_vi, world.goal, world.trap,
                               f"Gamma = {gamma}")

    fig.suptitle("Optimal State Values & Policy (Value Iteration) Across Discount Factors", fontsize=14)
    plt.tight_layout()
    plt.savefig("mdp_optimal_values.png", dpi=150)
    plt.close()
    print("State Values visual plot saved as 'mdp_optimal_values.png'.")

12
(1, 2)
--- Running Dynamic Programming for Gamma = 0.9 ---
Value Iteration converged in 52 iterations. Final delta: 8.39e-09
Policy Iteration converged in 6 steps (Policy is stable).
Max Value discrepancy between VI and PI: 8.68e-09

--- Running Dynamic Programming for Gamma = 0.95 ---
Value Iteration converged in 57 iterations. Final delta: 6.35e-09
Policy Iteration converged in 6 steps (Policy is stable).
Max Value discrepancy between VI and PI: 8.67e-09

--- Running Dynamic Programming for Gamma = 0.99 ---
Value Iteration converged in 61 iterations. Final delta: 7.95e-09
Policy Iteration converged in 6 steps (Policy is stable).
Max Value discrepancy between VI and PI: 1.19e-08

--- Running Dynamic Programming for Gamma = 0.999 ---
Value Iteration converged in 62 iterations. Final delta: 9.01e-09
Policy Iteration converged in 10 steps (Policy is stable).
Max Value discrepancy between VI and PI: 1.20e-08

State Values visual plot saved as 'mdp_optimal_values.png'.
